# Clase 226 — Federated learning manual (FedAvg)

Self-contained: solo numpy + sklearn. Simulamos FL **a mano** — sin `flower`, `PySyft` ni `TFF` — para que el algoritmo quede transparente. Seed 42.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=6,
    n_redundant=2, n_classes=2, random_state=42,
)
X = (X - X.mean(0)) / X.std(0)
# Columna de bias (sesgo) para regresión logística manual
X_b = np.hstack([X, np.ones((X.shape[0], 1))])
n, d = X_b.shape
print(f'dataset: X={X_b.shape}, y={y.shape}, balance={y.mean():.3f}')

## 1. Particionado IID en K=10 clientes

Shuffle uniforme. Cada cliente termina con ~200 muestras y clases balanceadas.

In [ ]:
K = 10

def split_iid(X, y, K, seed=42):
    idx = np.random.default_rng(seed).permutation(len(X))
    shards = np.array_split(idx, K)
    return [(X[s], y[s]) for s in shards]

clients_iid = split_iid(X_b, y, K)
for k, (Xk, yk) in enumerate(clients_iid):
    print(f'  client_{k}: n={len(Xk)}, pos_rate={yk.mean():.3f}')

## 2. Local training: regresión logística con SGD manual

Cada cliente recibe los pesos globales `w_t`, corre E épocas locales y devuelve `w_k`.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def loss_fn(w, X, y):
    p = sigmoid(X @ w)
    return -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))

def local_train(X, y, w_init, epochs=5, lr=0.05, batch_size=32, seed=0):
    w = w_init.copy()
    rng_local = np.random.default_rng(seed)
    for _ in range(epochs):
        idx = rng_local.permutation(len(X))
        for start in range(0, len(X), batch_size):
            b = idx[start:start + batch_size]
            p = sigmoid(X[b] @ w)
            grad = X[b].T @ (p - y[b]) / len(b)
            w -= lr * grad
    return w

## 3. FedAvg: agregación ponderada por tamaño del cliente

`w_{t+1} = Σ_k (n_k / n) · w_k`

In [ ]:
def fedavg(client_weights, client_sizes):
    total = sum(client_sizes)
    return sum((n_k / total) * w_k for w_k, n_k in zip(client_weights, client_sizes))


def run_fl(clients, rounds=20, clients_per_round=5, local_epochs=5, seed=42):
    rng_srv = np.random.default_rng(seed)
    w = np.zeros(d)
    history = []
    for t in range(rounds):
        sampled = rng_srv.choice(len(clients), clients_per_round, replace=False)
        weights, sizes = [], []
        for k in sampled:
            Xk, yk = clients[k]
            w_k = local_train(Xk, yk, w, epochs=local_epochs, seed=t * 100 + k)
            weights.append(w_k)
            sizes.append(len(Xk))
        w = fedavg(weights, sizes)
        history.append(loss_fn(w, X_b, y))
    return w, history

w_fl_iid, hist_iid = run_fl(clients_iid)
acc_fl_iid = accuracy_score(y, sigmoid(X_b @ w_fl_iid) > 0.5)
print(f'FedAvg IID — loss final={hist_iid[-1]:.4f}, accuracy={acc_fl_iid:.4f}')
print('curva (cada 4 rondas):', [round(hist_iid[i], 4) for i in range(0, 20, 4)])

## 4. Baseline centralizado (todo el data junto)

In [ ]:
central = LogisticRegression(max_iter=1000, C=1e6, solver='lbfgs').fit(X, y)
acc_central = central.score(X, y)
print(f'Central — accuracy={acc_central:.4f}')
print(f'Gap central vs FedAvg IID: {acc_central - acc_fl_iid:+.4f}')
print('  → en IID FedAvg recupera ~la misma performance que centralizar.')

## 5. Non-IID: cada cliente especializado

Asignamos a cada cliente principalmente una clase. FedAvg ya no converge limpio.

In [ ]:
def split_non_iid(X, y, K, alpha=0.1, seed=42):
    """Cada cliente recibe casi solo una clase (alpha = fracción de la 'otra')."""
    rng_s = np.random.default_rng(seed)
    idx_pos = rng_s.permutation(np.where(y == 1)[0])
    idx_neg = rng_s.permutation(np.where(y == 0)[0])
    shards = []
    half = K // 2
    pos_chunks = np.array_split(idx_pos, half)
    neg_chunks = np.array_split(idx_neg, K - half)
    for k in range(K):
        if k < half:
            main = pos_chunks[k]
            other = neg_chunks[k % len(neg_chunks)][:int(alpha * len(main))]
        else:
            main = neg_chunks[k - half]
            other = pos_chunks[(k - half) % len(pos_chunks)][:int(alpha * len(main))]
        ids = np.concatenate([main, other])
        shards.append((X[ids], y[ids]))
    return shards

clients_niid = split_non_iid(X_b, y, K)
for k, (Xk, yk) in enumerate(clients_niid):
    print(f'  client_{k}: n={len(Xk)}, pos_rate={yk.mean():.3f}')

In [ ]:
w_fl_niid, hist_niid = run_fl(clients_niid)
acc_fl_niid = accuracy_score(y, sigmoid(X_b @ w_fl_niid) > 0.5)
print(f'FedAvg non-IID — loss final={hist_niid[-1]:.4f}, accuracy={acc_fl_niid:.4f}')
print(f'\nIID  loss curve (cada 4): {[round(hist_iid[i], 4) for i in range(0, 20, 4)]}')
print(f'nIID loss curve (cada 4): {[round(hist_niid[i], 4) for i in range(0, 20, 4)]}')
print(f'\nDegradación accuracy IID → non-IID: {acc_fl_iid - acc_fl_niid:+.4f}')
print('  → con clientes especializados FedAvg oscila y converge peor (client drift).')

## 6. Gradient leakage didáctico

Demo simplificada de Zhu et al. 2019 (*Deep Leakage from Gradients*). Sobre **regresión lineal con batch=1**, el gradiente es:

`g = (w·x - y) · x`

Si el atacante conoce `w`, `y` y `g`, puede recuperar `x` minimizando `||(w·x̂ - y) · x̂ - g||²`. Mostramos que **compartir gradientes ≠ proteger data**.

In [ ]:
rng_l = np.random.default_rng(7)
w_true = rng_l.normal(0, 1, 10)
x_secret = rng_l.normal(0, 1, 10)
y_secret = float(w_true @ x_secret + rng_l.normal(0, 0.01))

# Gradiente compartido (lo único que el server ve en FL sin secure-agg)
g_shared = (w_true @ x_secret - y_secret) * x_secret

# Ataque: reconstruimos x_hat con SGD sobre el match-loss
x_hat = rng_l.normal(0, 1, 10)
for step in range(5000):
    resid = w_true @ x_hat - y_secret
    g_hat = resid * x_hat
    diff = g_hat - g_shared
    # ∂||g_hat - g||² / ∂x_hat
    grad_x = 2 * (diff @ w_true) * x_hat + 2 * resid * diff
    x_hat -= 0.01 * grad_x

mse = np.mean((x_hat - x_secret) ** 2)
print(f'x_secret  : {np.round(x_secret, 3)}')
print(f'x_reconstr: {np.round(x_hat, 3)}')
print(f'\nMSE de reconstrucción: {mse:.5f}')
print('  → el gradiente filtró la muestra. Defensas: secure aggregation + DP-FedAvg.')

## Ejercicio guiado

1. Implementá **FedProx**: en `local_train`, agregá `(mu/2) * ||w - w_global||²` al loss (gradiente extra: `mu * (w - w_global)`). Probá `mu ∈ {0.001, 0.01, 0.1}` sobre el setting non-IID.
2. Implementá **DP-FedAvg**: tras `fedavg`, sumá ruido `np.random.normal(0, sigma, d)` con `sigma=0.01`. Medí la caída de accuracy.
3. Variá `clients_per_round` ∈ {1, 3, 5, 10}. ¿Cuánto baja la varianza ronda a ronda?
4. En el ataque de leakage, probá `batch_size=8` (promediando 8 gradientes). ¿Sigue siendo reconstruible la primera muestra?
5. Cuantificá comunicación: `rondas × clientes_por_ronda × |w| × 4 bytes`. ¿Cuántos MB para 100 rondas con un modelo de 10M params?

## Conclusiones

- **FedAvg** es el baseline: muestreá clientes, entrená local E épocas, promediá pesos ponderados por `n_k`.
- En **IID** FedAvg converge cerca del centralizado; en **non-IID** degrada y oscila — usá FedProx o SCAFFOLD.
- **FL no es DP**: los gradientes filtran data (Zhu 2019). Producción real combina **FL + secure aggregation + DP**.
- Costo dominante: **comunicación**, no cómputo. Por eso compresión, quantization y top-k sparsification son áreas activas.
- En la práctica usá [Flower](https://flower.dev/), TFF o PySyft — pero el algoritmo es el de esta clase.